# Phase 3 — Orphan Taxa Search

Phase 1 scoped its reference database to **Chordopoxvirinae** (txid `10241`),
and Phase 2 searched it for A29L homologs. But Chordopoxvirinae and
Entomopoxvirinae are only two of the recognised subfamilies within
**Poxviridae** (txid `10240`); some poxvirus species are classified into
neither ("orphan" taxa). This phase searches that orphan set independently, so
a true homolog in an unclassified lineage is not missed merely for falling
outside the two named subfamilies.

> **Orphan taxa = Poxviridae species − Chordopoxvirinae species − Entomopoxvirinae species.**

All NCBI access reuses the shared `ncbi_fetch_utils` module — the same
implementation Phase 1 uses — so a fix to the retry, pagination, or taxonomy
logic applies everywhere at once.

**Steps**

0. Setup & configuration
1. Discover orphan taxa
2. Discover orphan protein IDs
3. Download orphan sequences
4. Search the orphan pool with jackhmmer
5. Extract orphan hit sequences

## Step 0 — Setup & configuration

`ncbi_fetch_utils` supplies `get_species_taxids`, `esearch_protein`,
`efetch_fasta`, and `FetchError`; `seqio_utils` supplies
`parse_fasta_by_accession` and `parse_hmmer_tabular`. Keep both modules beside
this notebook. `ncbi.configure()` registers the NCBI courtesy parameters once,
exactly as in Phase 1.

The query is **not** redefined here — this phase reuses `query.fasta` written
by **Phase 2**, so the orphan search targets the identical query. Its path is
set in `Config`; adjust it if your Phase 2 output folder differs.

**Paths and WSL.** As in Phase 2, any path passed to a `wsl`-prefixed command
uses `.as_posix()`, since Windows backslashes are literal characters inside
WSL. External tools are launched through the same `run_command` helper as
Phase 2, so failures raise a clear error and the exact command is logged.

In [1]:
from __future__ import annotations

import json
import logging
import os
import subprocess
import time
from dataclasses import dataclass, field
from pathlib import Path

import ncbi_fetch_utils as ncbi
from ncbi_fetch_utils import FetchError
from seqio_utils import parse_fasta_by_accession, parse_hmmer_tabular

In [2]:
@dataclass(frozen=True)
class Config:
    poxviridae_taxid: int = 10240
    chordopoxvirinae_taxid: int = 10241
    entomopoxvirinae_taxid: int = 10284

    min_len: int = 60
    max_len: int = 300
    ncbi_batch_size: int = 200
    max_ids_per_species: int = 400

    wsl_prefix: list[str] = field(default_factory=lambda: ["wsl"])
    num_iterations: int = 5
    evalue: str = "1e-3"
    inclusion_ethresh: str = "1e-3"

    phase2_query_fasta: Path = Path("output_phase2/query.fasta")

    out_dir: Path = field(default_factory=lambda: Path("output_phase3"))

    @property
    def orphan_taxids_path(self) -> Path:
        return self.out_dir / "orphan_taxids.json"

    @property
    def orphan_ids_path(self) -> Path:
        return self.out_dir / "orphan_ids.json"

    @property
    def orphan_fasta_path(self) -> Path:
        return self.out_dir / "orphan_proteins.fasta"

    @property
    def orphan_jack_tbl(self) -> Path:
        return self.out_dir / "orphan_jack.txt"

    @property
    def orphan_hits_fasta(self) -> Path:
        return self.out_dir / "orphan_hits.fasta"


CFG = Config()
CFG.out_dir.mkdir(parents=True, exist_ok=True)

ncbi.configure(
    email=os.environ.get("NCBI_EMAIL", ""),
    tool="poxvirus_pipeline",
    api_key=os.environ.get("NCBI_API_KEY", ""),
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(CFG.out_dir / "phase3.log"),
    ],
)
log = logging.getLogger("phase3")

if not CFG.phase2_query_fasta.exists():
    raise FileNotFoundError(
        f"{CFG.phase2_query_fasta} not found — run Phase 2 first, or update "
        "phase2_query_fasta in Config to point at the right file."
    )

In [3]:
class CommandError(RuntimeError):
    """Raised when an external tool exits with a non-zero status."""


def run_command(cmd: list[str], log_file: Path) -> None:
    log.info("Running: %s", " ".join(cmd))
    with open(log_file, "w", encoding="utf-8") as fh:
        result = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        raise CommandError(
            f"Command failed (exit {result.returncode}): {' '.join(cmd)}. "
            f"See {log_file} for details."
        )

## Step 1 — Discover orphan taxa

Enumerates species-level taxa under Poxviridae, Chordopoxvirinae, and
Entomopoxvirinae via the shared `get_species_taxids`, then subtracts the two
named subfamilies from the full family to obtain the orphan set. Checkpointed
on `orphan_taxids.json`.

In [4]:
if CFG.orphan_taxids_path.exists():
    log.info("%s exists — loading cached orphan taxids.", CFG.orphan_taxids_path)
    orphan_taxids = set(json.loads(CFG.orphan_taxids_path.read_text(encoding="utf-8")))
else:
    log.info("Enumerating Poxviridae, Chordopoxvirinae, and Entomopoxvirinae ...")
    poxviridae_sp = set(ncbi.get_species_taxids(CFG.poxviridae_taxid))
    chordo_sp = set(ncbi.get_species_taxids(CFG.chordopoxvirinae_taxid))
    entomo_sp = set(ncbi.get_species_taxids(CFG.entomopoxvirinae_taxid))

    orphan_taxids = poxviridae_sp - chordo_sp - entomo_sp

    log.info("Poxviridae: %d", len(poxviridae_sp))
    log.info("  minus Chordopoxvirinae: %d", len(chordo_sp))
    log.info("  minus Entomopoxvirinae: %d", len(entomo_sp))
    log.info("  = Orphan taxa: %d", len(orphan_taxids))

    CFG.orphan_taxids_path.write_text(json.dumps(sorted(orphan_taxids)), encoding="utf-8")

log.info("Orphan taxa to search: %d", len(orphan_taxids))

2026-07-22 11:24:26,402 [INFO] Enumerating Poxviridae, Chordopoxvirinae, and Entomopoxvirinae ...
2026-07-22 11:24:33,786 [INFO] Poxviridae: 353
2026-07-22 11:24:33,789 [INFO]   minus Chordopoxvirinae: 266
2026-07-22 11:24:33,790 [INFO]   minus Entomopoxvirinae: 47
2026-07-22 11:24:33,791 [INFO]   = Orphan taxa: 40
2026-07-22 11:24:33,795 [INFO] Orphan taxa to search: 40


## Step 2 — Discover orphan protein IDs

Searches NCBI Protein under each orphan taxid for sequences in the configured
length window via the shared `esearch_protein` — which paginates the full
result set, so orphan species with many records are not truncated. Checkpointed
on `orphan_ids.json`, kept separate from the FASTA download so a failure while
downloading does not force re-discovery of the IDs.

In [5]:
if CFG.orphan_ids_path.exists():
    log.info("%s exists — loading cached orphan protein IDs.", CFG.orphan_ids_path)
    orphan_ids = json.loads(CFG.orphan_ids_path.read_text(encoding="utf-8"))
else:
    log.info("Fetching orphan protein IDs ...")
    orphan_ids = []
    for i, tx in enumerate(sorted(orphan_taxids)):
        try:
            orphan_ids.extend(ncbi.esearch_protein(tx, min_len=CFG.min_len, max_len=CFG.max_len, max_ids_per_species=CFG.max_ids_per_species, logger=log))
        except FetchError as e:
            log.error("Skipping orphan taxid %s after repeated failures: %s", tx, e)
        if i % 25 == 0:
            log.info("taxon %d/%d — accumulated IDs: %d", i, len(orphan_taxids), len(orphan_ids))
        time.sleep(ncbi.rate_limit_s())

    orphan_ids = list(dict.fromkeys(orphan_ids))
    CFG.orphan_ids_path.write_text(json.dumps(orphan_ids), encoding="utf-8")

log.info("-> %d unique orphan protein IDs.", len(orphan_ids))

2026-07-22 11:24:34,081 [INFO] Fetching orphan protein IDs ...
2026-07-22 11:24:35,867 [INFO] taxon 0/40 — accumulated IDs: 0
2026-07-22 11:24:48,335 [INFO] taxid 1653087: capped at 400 IDs (1039 available)
2026-07-22 11:25:37,281 [INFO] taxon 25/40 — accumulated IDs: 2899
2026-07-22 11:26:13,467 [INFO] -> 3665 unique orphan protein IDs.


## Step 3 — Download orphan sequences

Fetches FASTA records in batches of `ncbi_batch_size` IDs via the shared
`efetch_fasta`, keeping each request under NCBI's URL-length limit — the same
batching rationale as Phase 1. Failed batches are logged and skipped.
Checkpointed on `orphan_proteins.fasta`.

In [6]:
if CFG.orphan_fasta_path.exists():
    log.info("%s exists — skipping orphan download.", CFG.orphan_fasta_path)
else:
    log.info("Downloading orphan sequences ...")
    with open(CFG.orphan_fasta_path, "w", encoding="utf-8") as out:
        for j in range(0, len(orphan_ids), CFG.ncbi_batch_size):
            chunk = orphan_ids[j:j + CFG.ncbi_batch_size]
            try:
                out.write(ncbi.efetch_fasta(chunk))
            except FetchError as e:
                log.error("Skipping ID chunk at %d: %s", j, e)
            time.sleep(ncbi.rate_limit_s())
    log.info("DONE — %s", CFG.orphan_fasta_path)

2026-07-22 11:26:13,545 [INFO] Downloading orphan sequences ...
2026-07-22 11:26:43,275 [INFO] DONE — output_phase3\orphan_proteins.fasta


## Step 4 — Search the orphan pool with jackhmmer

Runs the same query used in Phase 2 against the orphan protein pool with
`jackhmmer` only. PSI-BLAST and hmmsearch are not repeated here: Phase 2
established that all three methods agree closely on this query, so jackhmmer
alone suffices for this smaller, exploratory orphan check. (If a future query
needs all three for orphan taxa too, this step extends the same way Phase 2
does.) The search is launched through `run_command`, so a non-zero exit raises
a clear error and the command is logged. Checkpointed on `orphan_jack.txt`.

In [7]:
if CFG.orphan_jack_tbl.exists():
    log.info("%s exists — skipping orphan jackhmmer search.", CFG.orphan_jack_tbl)
else:
    log.info("Searching orphan pool with jackhmmer ...")
    run_command(
        CFG.wsl_prefix + [
            "jackhmmer", "-N", str(CFG.num_iterations),
            "-E", CFG.evalue, "--incE", CFG.inclusion_ethresh,
            "--tblout", CFG.orphan_jack_tbl.as_posix(),
            "-o", (CFG.out_dir / "orphan_jack.log").as_posix(),
            CFG.phase2_query_fasta.as_posix(), CFG.orphan_fasta_path.as_posix(),
        ],
        log_file=CFG.out_dir / "orphan_jack_run.log",
    )

orphan_hits = parse_hmmer_tabular(CFG.orphan_jack_tbl)
log.info("Orphan hits found: %d", len(orphan_hits))
for hit in sorted(orphan_hits):
    log.info("  -> %s", hit)

2026-07-22 11:26:43,331 [INFO] Searching orphan pool with jackhmmer ...
2026-07-22 11:26:43,333 [INFO] Running: wsl jackhmmer -N 5 -E 1e-3 --incE 1e-3 --tblout output_phase3/orphan_jack.txt -o output_phase3/orphan_jack.log output_phase2/query.fasta output_phase3/orphan_proteins.fasta
2026-07-22 11:26:44,277 [INFO] Orphan hits found: 3
2026-07-22 11:26:44,279 [INFO]   -> QDJ95055.1
2026-07-22 11:26:44,281 [INFO]   -> QHR82657.1
2026-07-22 11:26:44,283 [INFO]   -> WGO62741.1


## Step 5 — Extract orphan hit sequences

Pulls the sequences for the jackhmmer hits out of `orphan_proteins.fasta`
(via the shared `parse_fasta_by_accession`) and writes them to their own FASTA
file for downstream analysis.

In [8]:
orphan_db = parse_fasta_by_accession(CFG.orphan_fasta_path)

missing_from_db = sorted(acc for acc in orphan_hits if acc not in orphan_db)
if missing_from_db:
    log.warning(
        "%d orphan hit(s) not found in %s — check accession formatting: %s",
        len(missing_from_db), CFG.orphan_fasta_path, missing_from_db,
    )

with open(CFG.orphan_hits_fasta, "w", encoding="utf-8") as out:
    for acc in sorted(orphan_hits):
        seq = orphan_db.get(acc, "")
        if seq:
            out.write(f">{acc}\n{seq}\n")

log.info("DONE — orphan hit sequences written to %s", CFG.orphan_hits_fasta)

2026-07-22 11:26:44,361 [INFO] DONE — orphan hit sequences written to output_phase3\orphan_hits.fasta
